In [1]:
import networkx as nx

%load_ext autoreload
%autoreload 2

from utgen.raggraph.utils import get_node_context
from utgen.test_generation_crew.crew import TestGenerationCrew

In [2]:
# Load the GraphML file
g = nx.read_graphml("../data/repo_graph.graphml")

In [3]:
node_id = "utgen/raggraph/walker.py::function::iter_python_files"
node_id = "utgen/raggraph/utils.py::function::normalize_signature"

context = get_node_context(g=g, node_id=node_id)
print(context)

### NODE INFO ###
file::type::name -> utgen/raggraph/utils.py::function::normalize_signature

source:
def normalize_signature(node):
    """Return a Python-like signature including annotations and return type."""
    if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        return ""

    def unparse(x):
        try:
            return ast.unparse(x)
        except Exception:
            return ""

    parts = []

    # Pos-only args (Python 3.8+)
    for a in getattr(node.args, "posonlyargs", []):
        s = a.arg
        if a.annotation:
            s += f": {unparse(a.annotation)}"
        parts.append(s)
    if getattr(node.args, "posonlyargs", []):
        parts.append("/")

    # Regular args
    for a in node.args.args:
        s = a.arg
        if a.annotation:
            s += f": {unparse(a.annotation)}"
        parts.append(s)

    # Vararg
    if node.args.vararg:
        s = "*" + node.args.vararg.arg
        if node.args.vararg.annotation:
            s +

In [4]:
test_generator = TestGenerationCrew(guardrail_max_retries=3, verbose=True)

# Extract metadata
inputs = {"graph_context": context}

response = test_generator.crew().kickoff(inputs=inputs)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: Test generation crew                                                                                     │
│  ID: 37d6aaf3-9ce6-4b67-983e-ffd1b3978e10                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: generate_unit_tests_task                                                                                 │
│  ID: 8726d9ab-9d96-4a2a-affd-e80a59478495                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Context-Aware Test Architect                                                                            │
│                                                                                                                 │
│  Task: Generate a pytest suite for the function described in the context below.                                 │
│  CRITICAL INSTRUCTIONS: 1. **Target Code**: Focus on the source code under `### NODE INFO ###`. 2.              │
│  **Mocking**: Use `### OUTGOING EDGES ###` to identify dependencies.                                            │
│     Cross-reference with `### NEIGHBOR NODE DETAILS ###` to create                                              │
│     accurate `unittest.mock` objects that return the correct types.                                             │
│  3. **Input Selection (Incoming Edges)**: Analyze `### INCOMING EDGES ###`.                                     │
│     If other functions call this node, use those relationships to                                               │
│     infer realistic input values, common arguments, and the                                                     │
│     expected state of the system before the call.                                                               │
│  4. **Edge Cases**: Contrast the "intended use" (Incoming Edges)                                                │
│     with the "allowed use" (Signature/Types) to find boundary                                                   │
│     conditions and potential crash points.                                                                      │
│  5. **Naming**: Use the `file::type::name` to determine the correct                                             │
│     import paths for the test file.                                                                             │
│                                                                                                                 │
│  CONTEXT FROM GRAPH: ### NODE INFO ###                                                                          │
│  file::type::name -> utgen/raggraph/utils.py::function::normalize_signature                                     │
│                                                                                                                 │
│  source:                                                                                                        │
│  def normalize_signature(node):                                                                                 │
│      """Return a Python-like signature including annotations and return type."""                                │
│      if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):                                          │
│          return ""                                                                                              │
│                                                                                                                 │
│      def unparse(x):                                                                                            │
│          try:                                                                                                   │
│              return ast.unparse(x)                                                                              │
│          except Exception:                                                                                      │
│              return ""                                                                                          │
│                                                                                                                 │
│      parts = []                                        

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Context-Aware Test Architect                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```json                                                                                                        │
│  {                                                                                                              │
│    "tests": {                                                                                                   │
│      "test_normalize_signature_with_function_def": {                                                            │
│        "name": "test_normalize_signature_with_function_def",                                                    │
│        "imports": "import pytest\nfrom unittest.mock import Mock\nfrom utgen.raggraph.utils import              │
│  normalize_signature\nimport ast",                                                                              │
│        "code": "def test_normalize_signature_with_function_def():\n    # Create a mock FunctionDef node with    │
│  various argument types\n    mock_node = Mock(spec=ast.FunctionDef)\n    mock_node.name = 'test_function'\n     │
│  \n    # Mock args with posonlyargs, args, vararg, kwonlyargs, and kwarg\n    mock_args = Mock()\n              │
│  mock_args.posonlyargs = [\n        Mock(arg='a', annotation=ast.Name(id='int', ctx=ast.Load()))\n    ]\n       │
│  mock_args.args = [\n        Mock(arg='b', annotation=ast.Name(id='str', ctx=ast.Load()))\n    ]\n              │
│  mock_args.vararg = Mock(arg='args', annotation=ast.Name(id='tuple', ctx=ast.Load()))\n                         │
│  mock_args.kwonlyargs = [\n        Mock(arg='c', annotation=ast.Name(id='bool', ctx=ast.Load()))\n    ]\n       │
│  mock_args.kwarg = Mock(arg='kwargs', annotation=ast.Name(id='dict', ctx=ast.Load()))\n    mock_node.args =     │
│  mock_args\n    \n    # Mock return type\n    mock_node.returns = ast.Name(id='None', ctx=ast.Load())\n    \n   │
│  # Expected result\n    expected_signature = \"def test_function(a: int, /, b: str, *args: tuple, c: bool,      │
│  **kwargs: dict) -> None\"\n    \n    # Call the function\n    result = normalize_signature(mock_node)\n    \n  │
│  # Assert the result\n    assert result == expected_signature"                                                  │
│      },                                                                                                         │
│      "test_normalize_signature_with_async_function_def": {                                                      │
│        "name": "test_normalize_signature_with_async_function_def",                                              │
│        "imports": "import pytest\nfrom unittest.mock import Mock\nfrom utgen.raggraph.utils import              │
│  normalize_signature\nimport ast",                                                                              │
│        "code": "def test_normalize_signature_with_async_function_def():\n    # Create a mock AsyncFunctionDef   │
│  node\n    mock_node = Mock(spec=ast.AsyncFunctionDef)\n    mock_node.name = 'async_test_function'\n    \n      │
│  # Mock args with no arguments\n    mock_args = Mock()\n    mock_args.posonlyargs = []\n    mock_args.args =    │
│  []\n    mock_args.vararg = None\n    mock_args.kwonlyargs = []\n    mock_args.kwarg = None\n                   │
│  mock_node.args = mock_args\n    \n    # Mock return type\n    mock_node.returns =                              │
│  ast.Name(id='Awaitable[int]', ctx=ast.Load())\n    \n 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: generate_unit_tests_task                                                                                 │
│  Agent: Context-Aware Test Architect                                                                            │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: Test generation crew                                                                                     │
│  ID: 37d6aaf3-9ce6-4b67-983e-ffd1b3978e10                                                                       │
│  Final Output: ```json                                                                                          │
│  {                                                                                                              │
│    "tests": {                                                                                                   │
│      "test_normalize_signature_with_function_def": {                                                            │
│        "name": "test_normalize_signature_with_function_def",                                                    │
│        "imports": "import pytest\nfrom unittest.mock import Mock\nfrom utgen.raggraph.utils import              │
│  normalize_signature\nimport ast",                                                                              │
│        "code": "def test_normalize_signature_with_function_def():\n    # Create a mock FunctionDef node with    │
│  various argument types\n    mock_node = Mock(spec=ast.FunctionDef)\n    mock_node.name = 'test_function'\n     │
│  \n    # Mock args with posonlyargs, args, vararg, kwonlyargs, and kwarg\n    mock_args = Mock()\n              │
│  mock_args.posonlyargs = [\n        Mock(arg='a', annotation=ast.Name(id='int', ctx=ast.Load()))\n    ]\n       │
│  mock_args.args = [\n        Mock(arg='b', annotation=ast.Name(id='str', ctx=ast.Load()))\n    ]\n              │
│  mock_args.vararg = Mock(arg='args', annotation=ast.Name(id='tuple', ctx=ast.Load()))\n                         │
│  mock_args.kwonlyargs = [\n        Mock(arg='c', annotation=ast.Name(id='bool', ctx=ast.Load()))\n    ]\n       │
│  mock_args.kwarg = Mock(arg='kwargs', annotation=ast.Name(id='dict', ctx=ast.Load()))\n    mock_node.args =     │
│  mock_args\n    \n    # Mock return type\n    mock_node.returns = ast.Name(id='None', ctx=ast.Load())\n    \n   │
│  # Expected result\n    expected_signature = \"def test_function(a: int, /, b: str, *args: tuple, c: bool,      │
│  **kwargs: dict) -> None\"\n    \n    # Call the function\n    result = normalize_signature(mock_node)\n    \n  │
│  # Assert the result\n    assert result == expected_signature"                                                  │
│      },                                                                                                         │
│      "test_normalize_signature_with_async_function_def": {                                                      │
│        "name": "test_normalize_signature_with_async_function_def",                                              │
│        "imports": "import pytest\nfrom unittest.mock import Mock\nfrom utgen.raggraph.utils import              │
│  normalize_signature\nimport ast",                                                                              │
│        "code": "def test_normalize_signature_with_async_function_def():\n    # Create a mock AsyncFunctionDef   │
│  node\n    mock_node = Mock(spec=ast.AsyncFunctionDef)\n    mock_node.name = 'async_test_function'\n    \n      │
│  # Mock args with no arguments\n    mock_args = Mock()\n    mock_args.posonlyargs = []\n    mock_args.args =    │
│  []\n    mock_args.vararg = None\n    mock_args.kwonlyargs = []\n    mock_args.kwarg = None\n                   │
│  mock_node.args = mock_args\n    \n    # Mock return type\n    mock_node.returns =                              │
│  ast.Name(id='Awaitable[int]', ctx=ast.Load())\n    \n

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [5]:
print(response.raw)

```json
{
  "tests": {
    "test_normalize_signature_with_function_def": {
      "name": "test_normalize_signature_with_function_def",
      "imports": "import pytest\nfrom unittest.mock import Mock\nfrom utgen.raggraph.utils import normalize_signature\nimport ast",
      "code": "def test_normalize_signature_with_function_def():\n    # Create a mock FunctionDef node with various argument types\n    mock_node = Mock(spec=ast.FunctionDef)\n    mock_node.name = 'test_function'\n    \n    # Mock args with posonlyargs, args, vararg, kwonlyargs, and kwarg\n    mock_args = Mock()\n    mock_args.posonlyargs = [\n        Mock(arg='a', annotation=ast.Name(id='int', ctx=ast.Load()))\n    ]\n    mock_args.args = [\n        Mock(arg='b', annotation=ast.Name(id='str', ctx=ast.Load()))\n    ]\n    mock_args.vararg = Mock(arg='args', annotation=ast.Name(id='tuple', ctx=ast.Load()))\n    mock_args.kwonlyargs = [\n        Mock(arg='c', annotation=ast.Name(id='bool', ctx=ast.Load()))\n    ]\n    mock_ar

In [6]:
# Split into lines, slice from index 1 to -1, then rejoin
lines = response.raw.splitlines()
test_content = "\n".join(lines[1:-1])

print(test_content)

{
  "tests": {
    "test_normalize_signature_with_function_def": {
      "name": "test_normalize_signature_with_function_def",
      "imports": "import pytest\nfrom unittest.mock import Mock\nfrom utgen.raggraph.utils import normalize_signature\nimport ast",
      "code": "def test_normalize_signature_with_function_def():\n    # Create a mock FunctionDef node with various argument types\n    mock_node = Mock(spec=ast.FunctionDef)\n    mock_node.name = 'test_function'\n    \n    # Mock args with posonlyargs, args, vararg, kwonlyargs, and kwarg\n    mock_args = Mock()\n    mock_args.posonlyargs = [\n        Mock(arg='a', annotation=ast.Name(id='int', ctx=ast.Load()))\n    ]\n    mock_args.args = [\n        Mock(arg='b', annotation=ast.Name(id='str', ctx=ast.Load()))\n    ]\n    mock_args.vararg = Mock(arg='args', annotation=ast.Name(id='tuple', ctx=ast.Load()))\n    mock_args.kwonlyargs = [\n        Mock(arg='c', annotation=ast.Name(id='bool', ctx=ast.Load()))\n    ]\n    mock_args.kwarg

In [7]:
import json

# Convert string to dictionary
data = json.loads(test_content)

# Save to a file
with open('../data/tests_data.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=4)